[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/62_speculative_decoding_verification_solution.ipynb)

# 🔴 Solution: Speculative Decoding with Verification

Reference solution for `speculative_decoding_verification`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def _argmax_token(logits: torch.Tensor) -> int:
    return int(torch.argmax(logits, dim=-1).item())


def speculative_decode_verify(target_model, draft_model, prefix, max_new_tokens: int,
                              draft_steps: int = 4, eos_token_id: int | None = None):
    tokens = list(prefix)
    target_len = len(prefix) + max_new_tokens
    while len(tokens) < target_len:
        draft_context = list(tokens)
        proposed = []
        for _ in range(min(draft_steps, target_len - len(tokens))):
            tok = _argmax_token(draft_model(draft_context))
            proposed.append(tok)
            draft_context.append(tok)
            if eos_token_id is not None and tok == eos_token_id:
                break

        accepted_any = False
        for tok in proposed:
            target_tok = _argmax_token(target_model(tokens))
            if tok == target_tok:
                tokens.append(tok)
                accepted_any = True
                if eos_token_id is not None and tok == eos_token_id:
                    return tokens
            else:
                tokens.append(target_tok)
                if eos_token_id is not None and target_tok == eos_token_id:
                    return tokens
                break
            if len(tokens) >= target_len:
                break

        if not proposed or (not accepted_any and len(tokens) < target_len):
            target_tok = _argmax_token(target_model(tokens))
            tokens.append(target_tok)
            if eos_token_id is not None and target_tok == eos_token_id:
                return tokens
    return tokens


In [ ]:
# Verify
class IncrementModel:
    def __call__(self, tokens):
        logits = torch.zeros(10)
        logits[(tokens[-1] + 1) % 10] = 1
        return logits

print(speculative_decode_verify(IncrementModel(), IncrementModel(), [0], max_new_tokens=5, draft_steps=3))


In [ ]:
# Run judge
from torch_judge import check
check('speculative_decoding_verification')
